# التدريب والتقييم من A إلى Z
## زيد العنزي — Zaid Al-Anzi

دورة الذكاء الاصطناعي في قطاع الاتصالات. شغّل الخلايا بالتسلسل؛ CPU يكفي للمسار الأساسي. تجربة TensorFlow اختيارية.

**الهدف:** تدريب نماذج فعلية وتقييمها وحفظ الأدلة. البيانات تعليمية ولا تمثل شبكة تشغيلية.

الدليل: https://zaid.hs-rp.com/#/lab/colab


## 01. افتح الدفتر واحفظ نسختك

1. نزّل دفتر «التدريب والتقييم من A إلى Z» وملف CSV من زريّ التنزيل أعلاه.
2. افتح Google Colab وسجّل الدخول بحسابك. من File ← Upload notebook اختر ملف Telecom_Colab_A_to_Z.ipynb.
3. احفظ نسخة في Drive عبر File ← Save a copy in Drive إذا ظهر هذا الخيار. سمّها باسمك أو برمز الطالب، دون وضع كلمة مرور أو بيانات شخصية داخل الخلايا.
4. اختر Runtime ← Change runtime type ثم Python 3 وCPU. اضغط Connect. شغّل كل خلية بزر ▶ أو Shift+Enter، وانتظر اكتمالها قبل الخلية التالية.
5. المسار الأساسي يعمل على CPU. لتجربة TensorFlow يمكنك اختيار GPU إن كان متاحاً؛ نوعه وحصته يتغيران ولا يضمن Colab توفره. تغيير بيئة التشغيل قد يمحو متغيرات الجلسة: أعد التشغيل من البداية.

**تحقق:** تظهر علامة الاتصال، وتملك نسخة مستقلة من الدفتر. التدريب ينفّذ في حسابك في Colab، وليس في صفحة الموقع.


## 02. جهّز المكتبات وسجّل البيئة

1. شغّل الخلية التالية مرة واحدة. تثبت المكتبات الأساسية المفقودة فقط؛ لا تغيّر الإصدارات الموجودة عشوائياً.
2. احتفظ بالإصدارات المطبوعة مع تقريرك. إذا طلب Colab إعادة تشغيل الجلسة بعد التثبيت، نفّذها ثم شغّل الخلايا من أولها.
3. pandas للجداول، NumPy للمصفوفات، scikit-learn للتدريب والتقييم، Matplotlib للرسوم، وjoblib لحفظ النموذج. TensorFlow اختياري في خطوة مستقلة.

**تحقق:** تظهر إصدارات Python والمكتبات. لا يلزم تثبيت CUDA يدوياً داخل بيئة Colab المُدارة.


In [ ]:
import importlib.util, subprocess, sys, os, json, hashlib, platform, shutil
from pathlib import Path
from datetime import datetime, timezone
packages = {'numpy':'numpy', 'pandas':'pandas', 'sklearn':'scikit-learn',
            'matplotlib':'matplotlib', 'joblib':'joblib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, PrecisionRecallDisplay)
SEED = 42
np.random.seed(SEED)
VERSIONS = {'python':platform.python_version(), 'numpy':np.__version__,
            'pandas':pd.__version__, 'scikit-learn':sklearn.__version__}
print(json.dumps(VERSIONS, indent=2))


## 03. حمّل بيانات الدورة

1. إن رفعت CSV إلى لوحة Files في Colab، تستخدمه الخلية مباشرة. وإلا تحاول تنزيل النسخة الأصلية من موقع الدورة.
2. إذا تعذر الاتصال بالموقع، ستظهر نافذة رفع الملف في Colab. اختر Telecom_Network_Data.csv الذي نزّلته.
3. تتحقق الخلية من بصمة الملف لضمان أننا نعمل على المادة نفسها. لا تغيّر اسم عمود أو محتوى الملف في هذه التجربة.

**تحقق:** يظهر اسم الملف وبصمته ثم أول خمسة سجلات. أي خطأ في البصمة يعني أنك تحتاج النسخة الأصلية.


In [ ]:
from urllib.request import urlopen
DATA_URL = 'https://zaid.hs-rp.com/resources/Telecom_Network_Data.csv'
DATA_PATH = Path('Telecom_Network_Data.csv')
EXPECTED_SHA256 = 'fbcda0dafb37188791ed5c2aebea5dbbf1be3b33c4083b1a2969790140f13b01'
if not DATA_PATH.exists():
    try:
        with urlopen(DATA_URL, timeout=25) as response:
            DATA_PATH.write_bytes(response.read())
    except Exception as error:
        print('تعذر التنزيل. ارفع ملف CSV الأصلي من جهازك.')
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('ضع ملف CSV بجانب الدفتر ثم أعد تشغيل الخلية.') from error
        uploaded = files.upload()
        if DATA_PATH.name not in uploaded:
            raise ValueError('اختر الملف Telecom_Network_Data.csv بالتحديد.')
dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert dataset_sha256 == EXPECTED_SHA256, 'الملف مختلف. نزّل النسخة الأصلية من موقع الدورة.'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
print('SHA256:', dataset_sha256)
display(df.head())


## 04. افحص البيانات قبل التدريب

1. اقرأ شكل الجدول وتوزيع الهدف. عدد الصفوف المتوقع 3,605، وعدد الأبراج 5، والازدحام 203 حالات.
2. راقب القيم المفقودة والتكرار. وجود صف بلا ازدحام أكثر شيوعاً لا يعني أن اكتشاف الازدحام سهل أو أن الدقة مقياس كافٍ.
3. الملف تعليمي والهدف يطابق قاعدة حتمية. الرسوم هنا وصف لهذا الملف؛ لا تستنتج منها حالة شبكات العراق. لا توجد إحداثيات جغرافية في CSV.

**تحقق:** يعرض الرسم عدم توازن الفئات والعلاقة بين المستخدمين وزمن الاستجابة. غياب القيم المفقودة لا يثبت واقعية البيانات.


In [ ]:
assert len(df) == 3605 and df['tower_id'].nunique() == 5
assert df['congestion'].sum() == 203
assert not df.isna().any().any(), 'راجع القيم المفقودة قبل المتابعة.'
print('Shape:', df.shape, '| Duplicate rows:', int(df.duplicated().sum()))
display(df.isna().sum().to_frame('missing'))
display(df['congestion'].value_counts().sort_index().to_frame('rows'))
display(df[['users_connected','download_speed','upload_speed','latency']].describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df['congestion'].value_counts().sort_index().plot.bar(ax=axes[0], color=['#409b81','#d59c50'])
axes[0].set(title='Class balance', xlabel='Congestion', ylabel='Rows')
axes[1].scatter(df['users_connected'], df['latency'], c=df['congestion'], s=6, alpha=.45)
axes[1].axvline(800, ls='--', c='gray'); axes[1].axhline(150, ls='--', c='gray')
axes[1].set(xlabel='Connected users', ylabel='Latency (ms)', title='Educational data')
plt.tight_layout(); plt.show()


## 05. افصل التدريب والتحقق والاختبار

1. نستخدم 60% للتدريب، و20% للتحقق واختيار النموذج، و20% للاختبار النهائي. stratify يحافظ تقريباً على نسبة الفئات.
2. لا تُدخل congestion ضمن الخصائص. نحذف المعرّفات والزمن من هذا المثال، ونستخدم أربع خصائص عددية والطقس فقط.
3. هذا تقسيم عشوائي تعليمي لنفس المجموعة. لتقييم تنبؤ مستقبلي أو برج جديد يلزم تقسيم زمني أو بحسب البرج يناسب الاستخدام الفعلي.
4. من هذه النقطة لا تستخدم X_test أو y_test لاختيار نموذج أو عتبة. النتيجة النهائية تأتي بعد تثبيت الاختيار.

**تحقق:** تظهر 2,163 عينة تدريب و721 للتحقق و721 للاختبار، وفهارس منفصلة دون تداخل.


In [ ]:
NUMERIC = ['users_connected','download_speed','upload_speed','latency']
CATEGORICAL = ['weather']
FEATURES = NUMERIC + CATEGORICAL
X = df[FEATURES].copy()
y = df['congestion'].astype(int)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=.25, stratify=y_dev, random_state=SEED)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
print('Train:', len(X_train), '| Validation:', len(X_val), '| Test:', len(X_test))
for name, labels in [('Train',y_train),('Validation',y_val),('Test',y_test)]:
    print(name, 'positive share:', round(float(labels.mean()),4))


## 06. جهّز الخصائص دون تسرّب

1. نعوّض القيم العددية المفقودة بالوسيط ونوحّد مقياسها، ونحوّل الطقس إلى أعمدة One-Hot.
2. تُركّب المعالجة والنموذج داخل Pipeline. عند fit نتعلم التحويلات من التدريب فقط، ثم نطبّقها كما هي على التحقق والاختبار.
3. إن ظهر خطأ بأن make_preprocessor غير معرّفة، أعد تشغيل هذه الخلية ثم الخلية التي تحتاجها.

**تحقق:** تُعرّف دالة إنشاء المعالجة. لا تطبع هذه الخلية نتيجة تدريب؛ هذا متوقع.


In [ ]:
def make_preprocessor():
    numeric = Pipeline([('impute',SimpleImputer(strategy='median')), ('scale',StandardScaler())])
    categorical = Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                            ('encode',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([('numeric',numeric,NUMERIC),('weather',categorical,CATEGORICAL)])
def make_model(estimator):
    return Pipeline([('prepare',make_preprocessor()),('model',estimator)])
print('Pipeline ready: preprocessing will be fitted only on training rows.')


## 07. درّب خط الأساس وشجرة القرار

1. خط الأساس يتوقع الفئة الأكثر شيوعاً. هو مرجع للمقارنة وليس نموذجاً جيداً لاكتشاف الازدحام.
2. شجرة القرار تتعلم القواعد من أمثلة التدريب. class_weight يساعد على أخذ الفئة النادرة بالحسبان.
3. fit تعني التدريب، وpredict تعني إنتاج التصنيف. لا تخلط بينهما وبين قاعدة المنزلقات التوضيحية في الموقع.

**تحقق:** يظهر تأكيد تدريب النموذجين وعدد أمثلة التدريب، دون كشف نتائج الاختبار النهائي.


In [ ]:
models = {
    'baseline': make_model(DummyClassifier(strategy='most_frequent')),
    'decision_tree': make_model(DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=SEED))
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print('Trained:', name, '| rows:', len(X_train))


## 08. جرّب الغابة العشوائية على التحقق

1. جرّب ثلاثة أعماق للشجرة داخل غابة عشوائية. المقارنة هنا تستخدم مجموعة التحقق فقط.
2. نرتب المرشحين وفق F1 للتحقق. العتبة ثابتة 0.5. لا نرفع الأداء بإعادة التجربة على بيانات الاختبار.
3. AP هو Average Precision: تلخيص لمنحنى Precision–Recall، وليس مساحة محسوبة بالتكامل شبه المنحرف. كلاهما يختلف عن Accuracy.

**تحقق:** يظهر جدول مرشحي الغابة وF1 للتحقق، وجدول مقارنة أولي. التدريب لم يستخدم مجموعة الاختبار.


In [ ]:
validation_rows = []
def validation_metrics(name, probability):
    prediction = (probability >= .5).astype(int)
    return {'model':name, 'accuracy':float(accuracy_score(y_val,prediction)),
            'precision':float(precision_score(y_val,prediction,zero_division=0)),
            'recall':float(recall_score(y_val,prediction,zero_division=0)),
            'f1':float(f1_score(y_val,prediction,zero_division=0)),
            'average_precision':float(average_precision_score(y_val,probability))}
rf_candidates = []
for depth in [2,4,None]:
    candidate = make_model(RandomForestClassifier(n_estimators=100,max_depth=depth,
        class_weight='balanced',random_state=SEED,n_jobs=2))
    candidate.fit(X_train,y_train)
    score = validation_metrics('random_forest',candidate.predict_proba(X_val)[:,1])
    rf_candidates.append((score['f1'],candidate))
    print('Depth:',depth,'| Validation F1:',round(score['f1'],4))
models['random_forest'] = max(rf_candidates,key=lambda item:item[0])[1]
for name, model in models.items():
    validation_rows.append(validation_metrics(name,model.predict_proba(X_val)[:,1]))
display(pd.DataFrame(validation_rows).sort_values('f1',ascending=False).round(4))


## 09. جرّب TensorFlow وKeras

1. هذه تجربة إضافية مستقلة عن المسار الأساسي. غيّر RUN_TENSORFLOW إلى True لتفعيلها؛ اتركها False لإكمال التدريب الكلاسيكي.
2. TensorFlow إطار لبناء الشبكات العصبية، وKeras واجهته عالية المستوى. NVIDIA تصنع وحدات GPU؛ ليست مكتبة تدريب بديلة. CUDA طبقة لتشغيل حسابات GPU المتوافقة.
3. في Colab، اختر GPU من إعدادات Runtime إن توفر، ثم شغّل الدفتر من البداية. تعرض الخلية الأجهزة التي اكتشفها TensorFlow؛ القائمة الفارغة تعني التدريب على CPU.
4. إذا لم تكن TensorFlow مثبتة، شغّل %pip install tensorflow في خلية منفصلة، ثم أعد تشغيل الجلسة والخلايا. لا تثبّت تعريفات NVIDIA أو CUDA يدوياً لهذا الدفتر.
5. يستخدم EarlyStopping بيانات التحقق لإيقاف التدريب واستعادة أفضل أوزان. نتائج الشبكة قد تختلف بين البيئات حتى مع تثبيت البذرة.

**تحقق:** إن فُعّل المسار يظهر ملخص الشبكة ومنحنى loss وval_loss، ثم تُضاف TensorFlow إلى مقارنة التحقق. لا تحتاج GPU لهذا الجدول الصغير.


In [ ]:
RUN_TENSORFLOW = False  # Set True to run this optional experiment.
if RUN_TENSORFLOW:
    import tensorflow as tf
    from sklearn.utils.class_weight import compute_class_weight
    tf.keras.utils.set_random_seed(SEED)
    VERSIONS['tensorflow'] = tf.__version__
    print('TensorFlow:',tf.__version__,'| GPUs:',tf.config.list_physical_devices('GPU'))
    tf_prepare = make_preprocessor()
    train_array = tf_prepare.fit_transform(X_train).astype('float32')
    val_array = tf_prepare.transform(X_val).astype('float32')
    net = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(train_array.shape[1],)),
        tf.keras.layers.Dense(32,activation='relu'),
        tf.keras.layers.Dense(16,activation='relu'),
        tf.keras.layers.Dense(1,activation='sigmoid')
    ])
    net.compile(optimizer='adam',loss='binary_crossentropy',
                metrics=[tf.keras.metrics.AUC(curve='PR',name='pr_auc')])
    net.summary()
    weights = compute_class_weight(class_weight='balanced',classes=np.array([0,1]),y=y_train)
    class_weights = {i:float(v) for i,v in enumerate(weights)}
    train_ds = tf.data.Dataset.from_tensor_slices((train_array,y_train.to_numpy())).shuffle(len(y_train),seed=SEED).batch(64)
    val_ds = tf.data.Dataset.from_tensor_slices((val_array,y_val.to_numpy())).batch(64)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    train_ds, val_ds = train_ds.with_options(options), val_ds.with_options(options)
    history = net.fit(train_ds,validation_data=val_ds,epochs=35,class_weight=class_weights,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)],verbose=2)
    tf_val_probability = net(val_array,training=False).numpy().ravel()
    validation_rows = [r for r in validation_rows if r['model'] != 'tensorflow']
    validation_rows.append(validation_metrics('tensorflow',tf_val_probability))
    plt.figure(figsize=(7,3))
    plt.plot(history.history['loss'],label='Training loss')
    plt.plot(history.history['val_loss'],label='Validation loss')
    plt.xlabel('Epoch');plt.ylabel('Binary cross-entropy');plt.legend();plt.tight_layout();plt.show()
else:
    validation_rows = [r for r in validation_rows if r['model'] != 'tensorflow']
    print('Optional TensorFlow experiment skipped. The core course runs on CPU.')


## 10. ثبّت الاختيار ثم قيّم مرة واحدة

1. ثبّت أفضل نموذج حسب F1 للتحقق. بعد ذلك فقط افتح نتائج الاختبار. للمقارنة التعليمية نعرض جميع النماذج، لكن لا نعيد اختيار الأفضل بناءً على جدول الاختبار.
2. اقرأ Precision وRecall وF1 وAP ومصفوفة الالتباس، ولا تكتفِ بالدقة. حدود القرار 0.5 لجميع النماذج.
3. الاختبار جزء محفوظ من نفس البيانات التعليمية. الأداء العالي هنا لا يثبت النجاح على شبكة حقيقية أو على المستقبل.
4. إذا غيّرت الإعدادات بعد رؤية الاختبار، اعتبر هذه نتيجة استكشافية تحتاج بيانات اختبار جديدة لتأكيدها.

**تحقق:** يظهر اسم النموذج المختار، وجدول مقاييس الاختبار، ومصفوفة الالتباس ومنحنى Precision–Recall. خط الأساس يفوّت الازدحام؛ الدقة قد تبدو عالية.


In [ ]:
validation_table = pd.DataFrame(validation_rows).drop_duplicates('model',keep='last')
selected_model = str(validation_table.sort_values('f1',ascending=False,kind='stable').iloc[0]['model'])
print('Model selected using VALIDATION only:',selected_model)
def probabilities(name,frame):
    if name == 'tensorflow':
        return net(tf_prepare.transform(frame).astype('float32'),training=False).numpy().ravel()
    return models[name].predict_proba(frame)[:,1]
results, test_predictions = [], {}
for name in validation_table['model']:
    probability = probabilities(name,X_test)
    prediction = (probability >= .5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_test,prediction,labels=[0,1]).ravel()
    results.append({'model':name,'accuracy':float(accuracy_score(y_test,prediction)),
        'precision':float(precision_score(y_test,prediction,zero_division=0)),
        'recall':float(recall_score(y_test,prediction,zero_division=0)),
        'f1':float(f1_score(y_test,prediction,zero_division=0)),
        'average_precision':float(average_precision_score(y_test,probability)),
        'roc_auc':float(roc_auc_score(y_test,probability)),
        'confusion':{'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}})
    test_predictions[name] = {'probability':probability,'prediction':prediction}
display(pd.DataFrame(results).drop(columns='confusion').round(4))
selected = test_predictions[selected_model]
print(classification_report(y_test,selected['prediction'],target_names=['No congestion','Congestion'],zero_division=0))
OUTPUT = Path('telecom_results'); OUTPUT.mkdir(exist_ok=True)
fig,axes = plt.subplots(1,2,figsize=(11,4))
ConfusionMatrixDisplay.from_predictions(y_test,selected['prediction'],labels=[0,1],ax=axes[0],colorbar=False)
PrecisionRecallDisplay.from_predictions(y_test,selected['probability'],ax=axes[1],name=selected_model)
axes[0].set_title('Held-out confusion matrix');axes[1].set_title('Precision–Recall (test)')
plt.tight_layout();plt.savefig(OUTPUT/'evaluation.png',dpi=160,bbox_inches='tight');plt.show()


## 11. استعمل النموذج مع حالة جديدة

1. هذه حالة افتراضية يختبرها النموذج المختار. عدّل المستخدمين وزمن الاستجابة ثم أعد تشغيل هذه الخلية فقط.
2. القيمة المعادة درجة من النموذج وليست ضماناً أو احتمالاً تشغيلياً معايراً. قد يختلف النموذج عن القاعدة، خصوصاً قرب الحدود.
3. لا تعِد تدريب النموذج على كل حالة؛ predict أو استدعاء الشبكة يكفيان للاستدلال.

**تحقق:** تظهر درجة النموذج والقرار عند عتبة 0.5. هذا مثال على الاستدلال بعد التدريب.


In [ ]:
new_case = pd.DataFrame([{'users_connected':850,'download_speed':40.,
    'upload_speed':12.,'latency':170.,'weather':str(df['weather'].mode().iloc[0])}])[FEATURES]
score = float(probabilities(selected_model,new_case)[0])
display(new_case)
print('Model:',selected_model,'| Score:',round(score,4),'| Predicted congestion:',int(score>=.5))


## 12. احفظ النماذج والنتائج ونزّلها

1. تجمع الخلية جدول المقاييس والتنبؤات والصورة وبيانات البيئة في ملف ZIP. يحتوي metrics.json على نتائجك القابلة للعرض داخل الموقع.
2. تُحفظ نماذج scikit-learn كملفات joblib. إذا فعّلت TensorFlow يُحفظ النموذج بصيغة .keras مع معالجته الخاصة. لا تفتح ملفات نماذج من مصدر غير موثوق.
3. نزّل الأرشيف إلى جهازك، ثم احفظ الدفتر مع المخرجات عبر File ← Download ← Download .ipynb. ملفات جلسة Colab مؤقتة وقد تختفي عند انتهاء الجلسة.
4. على Colab يبدأ تنزيل ZIP تلقائياً. إن حظره المتصفح، افتح Files ثم قائمة ملف telecom_results.zip واختر Download.

**تحقق:** تحصل على telecom_results.zip. فك الضغط لعرض metrics.json وmetrics.csv وpredictions.csv وevaluation.png وrun.json وملفات النماذج.


In [ ]:
report = {'schema_version':1,'dataset_sha256':dataset_sha256,'test_rows':len(X_test),
          'selected_model':selected_model,'selection_basis':'validation_f1',
          'threshold':.5,'models':results}
(OUTPUT/'metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
pd.DataFrame(results).drop(columns='confusion').to_csv(OUTPUT/'metrics.csv',index=False)
validation_table.to_csv(OUTPUT/'validation_metrics.csv',index=False)
prediction_table = X_test.copy()
prediction_table.insert(0,'source_row',X_test.index)
prediction_table['actual'] = y_test
for name,pred in test_predictions.items():
    prediction_table[name+'_score'] = pred['probability']
    prediction_table[name+'_prediction'] = pred['prediction']
prediction_table.to_csv(OUTPUT/'predictions.csv',index=False)
for name,model in models.items():
    joblib.dump(model,OUTPUT/(name+'.joblib'))
if RUN_TENSORFLOW:
    net.save(OUTPUT/'tensorflow.keras')
    joblib.dump(tf_prepare,OUTPUT/'tensorflow_preprocessor.joblib')
run = {'created_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,'versions':VERSIONS,
       'train_rows':len(X_train),'validation_rows':len(X_val),'test_rows':len(X_test),
       'train_indices':X_train.index.tolist(),'validation_indices':X_val.index.tolist(),'test_indices':X_test.index.tolist(),
       'features':FEATURES,'dataset_sha256':dataset_sha256,'selected_model':selected_model,
       'limitation':'Educational deterministic labels; not evidence of operational generalization.'}
(OUTPUT/'run.json').write_text(json.dumps(run,indent=2),encoding='utf-8')
archive = shutil.make_archive('telecom_results','zip',OUTPUT)
print('Saved:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local Jupyter: download telecom_results.zip from your file browser.')


## 13. اعرض نتائجك وفسّرها

1. فك ضغط ZIP، ثم اختر metrics.json في لوحة «نتائجي من Colab» أسفل هذا الدليل. القراءة محلية في متصفحك ولا ترسل النتائج إلى خادم الموقع.
2. قارن خط الأساس بالنموذج المختار: كم حالة ازدحام اكتُشفت؟ كم حالة فاتت؟ وكم إنذاراً كان خاطئاً؟
3. Precision = TP/(TP+FP)، وRecall = TP/(TP+FN)، وF1 هو المتوسط التوافقي بينهما. نعرض صفراً عندما يكون المقام صفراً، كما في خط الأساس.
4. يمكن أن تختلف النتائج عن زميلك بسبب الإصدارات أو البذرة أو تفعيل TensorFlow. لا توجد درجة مضمونة أو شهادة اعتماد؛ اشرح الاختلاف بدلاً من نسخ نتيجة.
5. عند المشاركة، أرسل الدفتر المنفّذ والأرشيف والتفسير إلى المحاضر بالطريقة المتفق عليها خارج الموقع. افحص محتوى الدفتر قبل مشاركة رابط Drive واضبط الصلاحية إلى القراءة.

**تحقق:** اكتب استنتاجاً يذكر النموذج، تقسيم البيانات، المقاييس، كلفة الخطأ، وحدود تعميم النتيجة.


## 14. جهّز تسليمك الأكاديمي

1. أرفق نسخة الدفتر مع الخلايا المنفذة ومخرجاتها، وtelecom_results.zip، وفقرة تفسير من 150–250 كلمة.
2. تأكد أن train وvalidation وtest منفصلة، وأن الاختيار موثق قبل قراءة الاختبار، وأن الهدف غير موجود ضمن الخصائص.
3. وضّح أن البيانات تعليمية وأن القاعدة الحتمية لا تثبت أداءً إنتاجياً. اذكر إن فعّلت TensorFlow وعلى أي جهاز نفّذته.
4. قائمة مراجعة مقترحة للمحاضر: قابلية إعادة التشغيل 25%، سلامة المنهجية 30%، تفسير المقاييس 30%، توثيق الحدود والمصادر 15%. هذه معايير تعليمية وليست نتيجة تُحتسب آلياً في الموقع.

**تحقق:** لديك حزمة تدريب وتقييم كاملة يمكن إعادة تشغيلها ومناقشتها، مع تفسير علمي للنتائج.


## المصادر الرسمية

- https://research.google.com/colaboratory/faq.html
- https://scikit-learn.org/stable/modules/compose.html
- https://scikit-learn.org/stable/modules/model_evaluation.html
- https://www.tensorflow.org/guide/gpu
- https://keras.io/api/callbacks/early_stopping/
- https://www.tensorflow.org/tutorials/keras/save_and_load
